# 01 — Data Exploration

This notebook provides a quick-start environment for:
1. Pulling sample data from each API client.
2. Inspecting raw outputs.
3. Visualizing feature distributions and anomaly scores.

> **Note:** Make sure you have filled in your `.env` file with valid API keys before running live-data cells. Cells marked **(SYNTHETIC)** work without API keys.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid', palette='viridis')
%matplotlib inline

## 1. Synthetic Training Data (no API keys needed)

In [ ]:
from models.train_model import generate_training_data, FEATURE_NAMES

X, y = generate_training_data(n_normal=800, n_escalation=200)
df_train = pd.DataFrame(X, columns=FEATURE_NAMES)
df_train['label'] = y
df_train['label_str'] = df_train['label'].map({0: 'Normal', 1: 'Escalation'})

print(f'Training set shape: {df_train.shape}')
df_train.head()

In [ ]:
# Distribution of each feature by class
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, feat in zip(axes.flat, FEATURE_NAMES):
    for label, color in [(0, '#00b4d8'), (1, '#e63946')]:
        subset = df_train[df_train['label'] == label][feat]
        ax.hist(subset, bins=30, alpha=0.6, label='Normal' if label == 0 else 'Escalation', color=color)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)
plt.suptitle('Feature Distributions — Normal vs Escalation', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Feature Correlation Heatmap

In [ ]:
corr = df_train[FEATURE_NAMES].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Model Training & Evaluation

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

clf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42)
clf.fit(X_train_s, y_train)

y_pred = clf.predict(X_test_s)
print(classification_report(y_test, y_pred, target_names=['Normal', 'Escalation']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Escalation'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

## 4. Live API Data (requires API keys in `.env`)

Uncomment and run cells below once you have valid API keys.

In [ ]:
# from api_clients.gdelt_client import fetch_gdelt_articles
# articles_df = fetch_gdelt_articles(timespan='24h', max_records=20)
# articles_df[['timestamp', 'title', 'tone']].head(10)

In [ ]:
# from api_clients.finance_client import fetch_market_data
# market_df = fetch_market_data()
# market_df[['symbol', 'name', 'price', 'pct_change_24h', 'z_score']]